---
# 🎓 Proyecto de Machine Learning — Saber Pro
### Introducción a los Sistemas Inteligentes
**Corporación Universitaria Antonio José de Sucre — UAJS**

---

## 📌 Contexto
Clasificación binaria para identificar factores **sociodemográficos, socioeconómicos, laborales, de financiación e institucionales** asociados al rendimiento en Saber Pro (2018–2022).
- **0 = Bajo rendimiento** (≤ P20 del promedio de su año).
- **1 = Alto rendimiento** (≥ P80 de su año). Es la clase positiva.
- Se excluye la zona intermedia.

**Errores con esta convención:**
- **Falso positivo (FP):** el modelo dice *alto*, pero el estudiante era de bajo rendimiento. Es un estudiante en riesgo que no se detecta.
- **Falso negativo (FN):** el modelo dice *bajo*, pero el estudiante era de alto rendimiento. Es una alerta innecesaria.



> El proyecto identifica **asociaciones y patrones predictivos**, no relaciones causales.




---
# 📦 1. Instalación y carga de librerías
`xgboost` viene preinstalado en Google Colab.

In [ ]:
# ============================================================
# 📚 LIBRERÍAS
# ============================================================
import os, sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, RandomizedSearchCV, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, FunctionTransformer, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

from xgboost import XGBClassifier

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 200)

RANDOM_STATE = 42

# Para que los procesos paralelos de joblib encuentren features_saber_pro.py (se crea en la sección 7.0)
sys.path.insert(0, os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd() + os.pathsep + os.environ.get("PYTHONPATH", "")
print("✅ Librerías cargadas correctamente.")

---
# 📂 2. Carga del Dataset


In [ ]:
# ============================================================
# 📂 CARGA DEL DATASET
# ============================================================

# Se carga el archivo CSV desde los archivos de Google Colab.
dataset = pd.read_csv('Saber_Pro_muestra_20pct_multianual.csv', encoding='utf-8-sig')

# Se imprime confirmación y dimensiones del dataset.
print("✅ Dataset cargado correctamente.")
print(f"Filas: {dataset.shape[0]:,}")
print(f"Columnas: {dataset.shape[1]}")

# Se imprimen los nombres de todas las columnas.
print("\n📌 Columnas del dataset:")
print(dataset.columns.tolist())

# Se muestran las primeras 10 filas para una vista previa.
dataset.head(10)

---
# 🔍 3. Exploración inicial (EDA)


In [ ]:
# ============================================================
# 🗂️ GRUPOS DE COLUMNAS
# ============================================================
COL_ID = 'ESTU_CONSECUTIVO'
COL_ANIO = 'AÑO'
COL_PUNTAJE = 'PUNTAJE_PROMEDIO'

CATEGORICAS = [
    'ESTU_DEPTO_RESIDE', 'ESTU_GENERO', 'ESTU_HORASSEMANATRABAJA',
    'FAMI_EDUCACIONPADRE', 'FAMI_EDUCACIONMADRE', 'FAMI_ESTRATOVIVIENDA',
    'FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA', 'FAMI_TIENECOMPUTADOR', 'FAMI_TIENEINTERNET',
    'ESTU_VALORMATRICULAUNIVERSIDAD',
    'ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
    'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO',
    'INST_CARACTER_ACADEMICO', 'INST_ORIGEN', 'ESTU_METODO_PRGM',
]
NUMERICAS = ['EDAD']
PREDICTORAS = CATEGORICAS + NUMERICAS
# Columnas que vienen ya calculadas en el archivo a partir del puntaje: NO se usan como predictores
DERIVADAS_DEL_PUNTAJE = ['PUNTAJE_PROMEDIO', 'UMBRAL_P25', 'BAJO_RENDIMIENTO']

faltan = set(PREDICTORAS + DERIVADAS_DEL_PUNTAJE + [COL_ID, COL_ANIO]) - set(dataset.columns)
print("✅ Columnas esperadas presentes." if not faltan else f"❌ Faltan: {faltan}")

## 3.1 Estructura, faltantes, duplicados y columnas constantes

In [ ]:
display(pd.DataFrame({
    'tipo'          : dataset.dtypes.astype(str),
    'únicos'        : dataset.nunique(),
    'faltantes'     : dataset.isna().sum(),
    '% faltantes'   : (dataset.isna().mean() * 100).round(2),
}))

print(f"Filas duplicadas exactas      : {dataset.duplicated().sum():,}")
print(f"Identificadores repetidos     : {dataset[COL_ID].duplicated().sum():,}")
constantes = [c for c in dataset.columns if dataset[c].nunique(dropna=False) == 1]
print(f"Columnas constantes           : {constantes}")

print("\n📌 Registros por año:")
display(dataset[COL_ANIO].value_counts().sort_index().to_frame('registros'))

## 3.2 Problemas de codificación de texto

In [ ]:
texto = [c for c in CATEGORICAS if dataset[c].dtype != 'float64']
danados = {c: dataset[c].dropna().astype(str).str.contains('Ã|Â|�').sum() for c in texto}
display(pd.Series(danados, name='valores con caracteres dañados').to_frame().query('`valores con caracteres dañados` > 0'))
print("Ejemplos:", dataset['INST_ORIGEN'].unique()[:4])

## 3.3 Puntaje promedio y edad

In [ ]:
display(dataset[[COL_PUNTAJE, 'EDAD']].describe(percentiles=[.001, .01, .2, .5, .8, .99]).round(1).T)
print(f"Estudiantes con puntaje promedio = 0: {(dataset[COL_PUNTAJE] == 0).sum()}")
print(f"Edades fuera de 15–80 años          : {((dataset['EDAD'] < 15) | (dataset['EDAD'] > 80)).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(dataset[COL_PUNTAJE], bins=100, ax=axes[0], color='#4c72b0')
axes[0].set_title('Puntaje promedio')
sns.boxplot(x=dataset[COL_ANIO].astype(str), y=dataset[COL_PUNTAJE], ax=axes[1], color='#dd8452', fliersize=1)
axes[1].set_title('Puntaje promedio por año')
plt.tight_layout(); plt.show()

## 3.4 ¿Qué variables se relacionan más con el rendimiento?
**η² (eta cuadrado):** proporción de la variabilidad del puntaje explicada por cada variable por separado. Para la edad se agrupa en rangos.

In [ ]:
def eta_cuadrado(categoria, valores):
    g = categoria.astype(str)
    media = valores.mean()
    entre = valores.groupby(g).apply(lambda s: len(s) * (s.mean() - media) ** 2).sum()
    return entre / ((valores - media) ** 2).sum()

eta = {c: eta_cuadrado(dataset[c].fillna('(faltante)'), dataset[COL_PUNTAJE]) for c in CATEGORICAS}
eta['EDAD (rangos)'] = eta_cuadrado(pd.cut(dataset['EDAD'], [14, 21, 24, 28, 35, 80]), dataset[COL_PUNTAJE])
eta = pd.Series(eta).sort_values()

plt.figure(figsize=(9, 7))
plt.barh(eta.index, eta.values, color='#55a868')
plt.xlabel('η² (variabilidad del puntaje explicada)')
plt.title('Asociación de cada variable con el puntaje promedio', fontweight='bold')
plt.tight_layout(); plt.show()

---
# ⚙️ 4. Preprocesamiento de Datos
## 4.1 Corrección de codificación
Algunas columnas tienen caracteres dañados (por ejemplo, `INSTITUCIÃ\x93N`), producto de guardar el archivo con una codificación distinta a UTF-8. Se reparan reinterpretando los bytes; si no se corrigen, el mismo valor quedaría como dos categorías distintas.

In [ ]:
# ============================================================
# 4.1 CORRECCIÓN DE CODIFICACIÓN
# ============================================================
def reparar_texto(valor):
    if not isinstance(valor, str):
        return valor
    for codificacion in ("cp1252", "latin-1"):
        try:
            return valor.encode(codificacion).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return valor

columnas_texto = [c for c in dataset.columns
                  if not pd.api.types.is_numeric_dtype(dataset[c]) and c != COL_ID]
for col in columnas_texto:
    mapa = {v: reparar_texto(v) for v in dataset[col].dropna().unique()}
    dataset[col] = dataset[col].map(mapa)

restantes = sum(dataset[c].dropna().astype(str).str.contains('Ã|Â|�').sum() for c in columnas_texto)
print(f"✅ Corrección aplicada. Valores dañados restantes: {restantes}")
print(dataset['INST_ORIGEN'].unique())

## 4.2 Limpieza de datos

| Decisión | Justificación |
|---|---|
| Eliminar `ESTU_CONSECUTIVO` como predictor | Es un identificador, no describe al estudiante. |
| Eliminar `ESTU_NIVEL_PRGM_ACADEMICO` | Es constante (`UNIVERSITARIO`). |
| No usar `UMBRAL_P25` ni el `BAJO_RENDIMIENTO` del archivo | Se calcularon con el puntaje; la etiqueta se reconstruye en la sección 5. |
| Excluir estudiantes con promedio = 0 | Un promedio de 0 significa que los 5 módulos están en 0. Según el ICFES, un 0 indica módulo sin calificación (respuesta no registrada o no calificable), no un desempeño real. |
| `ESTU_HORASSEMANATRABAJA = '0'` → `'No trabaja'` | Es la misma respuesta escrita como texto, para que sea legible. |
| `DISTANCIA VITUAL` → `DISTANCIA VIRTUAL` | Error tipográfico de la fuente. |
| Faltantes categóricos → `Sin información` | Los faltantes pueden contener información; imputar la moda la borraría. Es un reemplazo fijo, sin fuga. |
| Edad faltante | Se imputa con la mediana **dentro del Pipeline** (aprendida solo con entrenamiento). |



In [ ]:
# ============================================================
# 4.2 LIMPIEZA
# ============================================================
n_inicial = len(dataset)

dataset = dataset.drop(columns=['ESTU_NIVEL_PRGM_ACADEMICO'])
dataset = dataset[dataset[COL_PUNTAJE] > 0].reset_index(drop=True)
print(f"Estudiantes excluidos por promedio = 0: {n_inicial - len(dataset)}")

dataset['ESTU_HORASSEMANATRABAJA'] = dataset['ESTU_HORASSEMANATRABAJA'].replace({'0': 'No trabaja'})
dataset['ESTU_METODO_PRGM'] = dataset['ESTU_METODO_PRGM'].replace({'DISTANCIA VITUAL': 'DISTANCIA VIRTUAL'})

CAT_FALTANTE = 'Sin información'
dataset[CATEGORICAS] = dataset[CATEGORICAS].fillna(CAT_FALTANTE)

print(f"✅ Limpieza completada → {len(dataset):,} estudiantes")
print(f"Faltantes en categóricas: {dataset[CATEGORICAS].isna().sum().sum()} | "
      f"faltantes en EDAD: {dataset['EDAD'].isna().sum()} (se imputan en el Pipeline)")

---
# 🎯 5. Construcción de la variable objetivo
Igual que en la entrega anterior, para que los resultados sean comparables:
- Para cada año se calcula el **P20** y el **P80** del puntaje promedio.
- Por debajo del P20 → **0 (bajo rendimiento)**; por encima del P80 → **1 (alto rendimiento)**.
- Los cortes se calculan **por año** porque las aplicaciones de 2018 a 2022 tuvieron condiciones distintas (por ejemplo, la pandemia en 2020–2021). Así, cada estudiante se compara con quienes presentaron la prueba en su mismo año.

In [ ]:
# ============================================================
# 5. VARIABLE OBJETIVO
# ============================================================
Q_BAJO, Q_ALTO = 0.20, 0.80

dataset['CORTE_BAJO'] = dataset.groupby(COL_ANIO)[COL_PUNTAJE].transform(lambda s: s.quantile(Q_BAJO))
dataset['CORTE_ALTO'] = dataset.groupby(COL_ANIO)[COL_PUNTAJE].transform(lambda s: s.quantile(Q_ALTO))

es_bajo = dataset[COL_PUNTAJE] <= dataset['CORTE_BAJO']
es_alto = dataset[COL_PUNTAJE] >= dataset['CORTE_ALTO']
dataset_modelo = dataset[es_bajo | es_alto].copy()
dataset_modelo['Y_ALTO'] = (dataset_modelo[COL_PUNTAJE] >= dataset_modelo['CORTE_ALTO']).astype(int)

ETIQUETAS = {0: 'Bajo rendimiento', 1: 'Alto rendimiento'}
print(f"Estudiantes en los extremos: {len(dataset_modelo):,} (zona intermedia excluida: {len(dataset) - len(dataset_modelo):,})")
display(dataset_modelo['Y_ALTO'].map({0: 'Bajo (0)', 1: 'Alto (1)'}).value_counts().to_frame('n'))
display(dataset.groupby(COL_ANIO)[['CORTE_BAJO', 'CORTE_ALTO']].first().round(1))

---
# 🧩 6. Selección de variables (X e y)
19 predictores: 18 categóricos y la edad. Quedan fuera el identificador, el año (por construcción, todos los años tienen la misma proporción de clases) y todo lo derivado del puntaje.

In [ ]:
X = dataset_modelo[PREDICTORAS].copy()
y = dataset_modelo['Y_ALTO'].copy()
anio_modelo = dataset_modelo[COL_ANIO].copy()     # NO es predictor

prohibidas = set(DERIVADAS_DEL_PUNTAJE + ['CORTE_BAJO', 'CORTE_ALTO', 'Y_ALTO', COL_ID, COL_ANIO])
assert prohibidas.isdisjoint(X.columns), "❌ Fuga de información en X"
print(f"✅ X: {X.shape[0]:,} × {X.shape[1]} | y: {y.value_counts().to_dict()} (0 = bajo, 1 = alto)")

---
# 🔀 7. División entrenamiento / prueba y validación
- **80/20 estratificado** por clase y año. La prueba queda sellada hasta la sección 9.
- **Validación cruzada estratificada de 5 particiones** (las mismas para todos los modelos) para ajustar hiperparámetros.
- **Probabilidades out-of-fold** en entrenamiento para elegir el umbral, la zona de incertidumbre y el modelo ganador.

In [ ]:
X_train, X_test, y_train, y_test, anio_train, anio_test = train_test_split(
    X, y, anio_modelo, test_size=0.20, random_state=RANDOM_STATE,
    stratify=y.astype(str) + '_' + anio_modelo.astype(str))

print(f"Entrenamiento: {len(X_train):,} | Prueba: {len(X_test):,}")
print(f"% alto → train {y_train.mean()*100:.2f} | test {y_test.mean()*100:.2f}")
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 7.0 Variables derivadas
Se crean con una función **fila a fila** que usa solo las 19 respuestas del estudiante: no aprende nada de los datos, así que no hay fuga de información. Se guarda en `features_saber_pro.py` porque la app de Streamlit necesita el mismo archivo para cargar los modelos.

| Grupo | Variables nuevas |
|---|---|
| Resúmenes numéricos | `N_BIENES`, `N_FUENTES_PAGO`, `EDU_MAX_PADRES`, `EDU_MIN_PADRES`, `ESTRATO_NUM`, `HORAS_NUM`, `MATRICULA_NUM`, `EDAD_DIF25_CUAD` |
| Rango de edad | `EDAD_RANGO` |
| Cruces | `DEPTO_x_ORIGEN`, `CARACTER_x_ORIGEN`, `ORIGEN_x_METODO`, `ORIGEN_x_MATRICULA`, `DEPTO_x_ESTRATO` |

In [ ]:
%%writefile features_saber_pro.py
# ============================================================
# Variables derivadas del proyecto Saber Pro
# ============================================================
# Este archivo se usa en el notebook Y en la app de Streamlit. Los Pipelines guardados lo referencian
# por nombre, así que debe estar en la misma carpeta que app.py.
# Solo usa las 19 variables originales de UN estudiante (fila a fila): no aprende nada de los datos,
# por lo que no puede producir fuga de información.
import numpy as np
import pandas as pd

BIENES = ['FAMI_TIENEAUTOMOVIL', 'FAMI_TIENELAVADORA', 'FAMI_TIENECOMPUTADOR', 'FAMI_TIENEINTERNET']
PAGOS = ['ESTU_PAGOMATRICULABECA', 'ESTU_PAGOMATRICULACREDITO',
         'ESTU_PAGOMATRICULAPADRES', 'ESTU_PAGOMATRICULAPROPIO']

RANGO_EDUCACION = {'Ninguno': 0, 'Primaria incompleta': 1, 'Primaria completa': 2,
                   'Secundaria (Bachillerato) incompleta': 3, 'Secundaria (Bachillerato) completa': 4,
                   'Técnica o tecnológica incompleta': 5, 'Técnica o tecnológica completa': 6,
                   'Educación profesional incompleta': 7, 'Educación profesional completa': 8, 'Postgrado': 9}
RANGO_MATRICULA = {'No pagó matrícula': 0, 'Menos de 500 mil': 1, 'Entre 500 mil y menos de 1 millón': 2,
                   'Entre 1 millón y menos de 2.5 millones': 3, 'Entre 2.5 millones y menos de 4 millones': 4,
                   'Entre 4 millones y menos de 5.5 millones': 5, 'Entre 5.5 millones y menos de 7 millones': 6,
                   'Más de 7 millones': 7}
RANGO_HORAS = {'No trabaja': 0, 'Menos de 10 horas': 1, 'Entre 11 y 20 horas': 2,
               'Entre 21 y 30 horas': 3, 'Más de 30 horas': 4}

NUM_DERIVADAS = ['N_BIENES', 'N_FUENTES_PAGO', 'EDU_MAX_PADRES', 'EDU_MIN_PADRES', 'ESTRATO_NUM',
                 'HORAS_NUM', 'MATRICULA_NUM', 'EDAD_DIF25_CUAD']
CAT_DERIVADAS = ['EDAD_RANGO', 'DEPTO_x_ORIGEN', 'CARACTER_x_ORIGEN', 'ORIGEN_x_METODO',
                 'ORIGEN_x_MATRICULA', 'DEPTO_x_ESTRATO']


def _es_si(serie):
    t = serie.astype(str).str.strip().str.lower().str.replace('í', 'i', regex=False)
    return (t == 'si').astype(float)


def _cruce(a, b):
    return a.astype(str) + ' | ' + b.astype(str)


def agregar_variables(X):
    """Devuelve una copia de X con las variables derivadas añadidas (las 19 originales se conservan)."""
    X = X.copy()
    X['N_BIENES'] = sum(_es_si(X[c]) for c in BIENES)
    X['N_FUENTES_PAGO'] = sum(_es_si(X[c]) for c in PAGOS)

    padre = X['FAMI_EDUCACIONPADRE'].map(RANGO_EDUCACION).astype(float)
    madre = X['FAMI_EDUCACIONMADRE'].map(RANGO_EDUCACION).astype(float)
    X['EDU_MAX_PADRES'] = np.fmax(padre, madre)
    X['EDU_MIN_PADRES'] = np.fmin(padre, madre)

    X['ESTRATO_NUM'] = X['FAMI_ESTRATOVIVIENDA'].astype(str).str.extract(r'(\d)')[0].astype(float)
    X['HORAS_NUM'] = X['ESTU_HORASSEMANATRABAJA'].map(RANGO_HORAS).astype(float)
    X['MATRICULA_NUM'] = X['ESTU_VALORMATRICULAUNIVERSIDAD'].map(RANGO_MATRICULA).astype(float)

    edad = pd.to_numeric(X['EDAD'], errors='coerce')
    X['EDAD_DIF25_CUAD'] = (edad - 25) ** 2          # efecto no lineal de la edad (para el modelo lineal)
    rango = pd.cut(edad, [0, 21, 24, 28, 35, 120], labels=['<=21', '22-24', '25-28', '29-35', '>35'])
    X['EDAD_RANGO'] = rango.astype(object).fillna('Sin información')

    # Cruces: el efecto de una variable cambia según otra (p. ej., la región depende del tipo de institución)
    X['DEPTO_x_ORIGEN'] = _cruce(X['ESTU_DEPTO_RESIDE'], X['INST_ORIGEN'])
    X['CARACTER_x_ORIGEN'] = _cruce(X['INST_CARACTER_ACADEMICO'], X['INST_ORIGEN'])
    X['ORIGEN_x_METODO'] = _cruce(X['INST_ORIGEN'], X['ESTU_METODO_PRGM'])
    X['ORIGEN_x_MATRICULA'] = _cruce(X['INST_ORIGEN'], X['ESTU_VALORMATRICULAUNIVERSIDAD'])
    X['DEPTO_x_ESTRATO'] = _cruce(X['ESTU_DEPTO_RESIDE'], X['FAMI_ESTRATOVIVIENDA'])
    return X


In [ ]:
import importlib, features_saber_pro
importlib.reload(features_saber_pro)
from features_saber_pro import agregar_variables, NUM_DERIVADAS, CAT_DERIVADAS

X_demo = agregar_variables(X_train.head(5))
print(f"✅ Variables derivadas listas: {len(NUM_DERIVADAS)} numéricas + {len(CAT_DERIVADAS)} categóricas "
      f"(de {X_train.shape[1]} a {X_demo.shape[1]} columnas).")
assert set(X_train.columns) <= set(X_demo.columns), "❌ Se perdieron columnas originales"
display(X_demo[NUM_DERIVADAS + CAT_DERIVADAS].head(3).T)

## 7.1 Transformación de datos por modelo (completa la sección 4.3)

| Modelo | Categóricas | Numéricas | ¿Por qué? |
|---|---|---|---|
| Regresión Logística | One-hot (categorías con < 30 casos se agrupan) + los **cruces** | Mediana + `StandardScaler` | Un modelo lineal necesita columnas 0/1 y escalas comparables. Los cruces le permiten aprender interacciones. |
| Decision Tree, Random Forest, XGBoost | Ordinal con orden de dominio (estrato, educación, horas, matrícula); ordinal simple en variables de pocas categorías; ***target encoding*** en departamento, carácter, origen, metodología y los cruces | Mediana | Los árboles dividen por umbrales y no necesitan escalamiento. El *target encoding* convierte categorías numerosas en una sola columna ordenada por su tasa de alto rendimiento (con validación cruzada interna). |

Todos los preprocesadores, **incluidas las variables derivadas**, van dentro de un `Pipeline`: se ajustan solo con los datos de entrenamiento de cada partición.

In [ ]:
# ============================================================
# 7.1 PREPROCESADORES
# ============================================================
ORDEN_ESTRATO = ['Estrato 1', 'Estrato 2', 'Estrato 3', 'Estrato 4', 'Estrato 5', 'Estrato 6', 'Sin Estrato', CAT_FALTANTE]
ORDEN_EDUCACION = ['Ninguno', 'Primaria incompleta', 'Primaria completa',
                   'Secundaria (Bachillerato) incompleta', 'Secundaria (Bachillerato) completa',
                   'Técnica o tecnológica incompleta', 'Técnica o tecnológica completa',
                   'Educación profesional incompleta', 'Educación profesional completa',
                   'Postgrado', 'No sabe', 'No Aplica', CAT_FALTANTE]
ORDEN_HORAS = ['No trabaja', 'Menos de 10 horas', 'Entre 11 y 20 horas', 'Entre 21 y 30 horas',
               'Más de 30 horas', CAT_FALTANTE]
ORDEN_MATRICULA = ['No pagó matrícula', 'Menos de 500 mil', 'Entre 500 mil y menos de 1 millón',
                   'Entre 1 millón y menos de 2.5 millones', 'Entre 2.5 millones y menos de 4 millones',
                   'Entre 4 millones y menos de 5.5 millones', 'Entre 5.5 millones y menos de 7 millones',
                   'Más de 7 millones', CAT_FALTANTE]
ORDENES = {'FAMI_ESTRATOVIVIENDA': ORDEN_ESTRATO, 'FAMI_EDUCACIONPADRE': ORDEN_EDUCACION,
           'FAMI_EDUCACIONMADRE': ORDEN_EDUCACION, 'ESTU_HORASSEMANATRABAJA': ORDEN_HORAS,
           'ESTU_VALORMATRICULAUNIVERSIDAD': ORDEN_MATRICULA}
CAT_CON_ORDEN = list(ORDENES)
CAT_SIN_ORDEN = [c for c in CATEGORICAS if c not in ORDENES]

for col, orden in ORDENES.items():
    sin_orden = set(X[col].unique()) - set(orden)
    assert not sin_orden, f"❌ {col}: categorías sin orden definido → {sin_orden}"

CAT_TE = ['ESTU_DEPTO_RESIDE', 'INST_CARACTER_ACADEMICO', 'INST_ORIGEN', 'ESTU_METODO_PRGM']   # muchas categorías / institucionales
CAT_NOM = [c for c in CATEGORICAS if c not in ORDENES and c not in CAT_TE]
NUM_TODAS = NUMERICAS + NUM_DERIVADAS

def nuevo_fe():
    return FunctionTransformer(agregar_variables)      # añade las variables derivadas dentro del Pipeline

def pipe_num(escalar=False):
    pasos = [('imp', SimpleImputer(strategy='median'))] + ([('esc', StandardScaler())] if escalar else [])
    return Pipeline(pasos)

preproc_lr = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=30), CATEGORICAS + CAT_DERIVADAS),
    ('num', pipe_num(escalar=True), NUM_TODAS),
])

preproc_ordinal = ColumnTransformer([
    ('ord', OrdinalEncoder(categories=[ORDENES[c] for c in CAT_CON_ORDEN],
                           handle_unknown='use_encoded_value', unknown_value=-1), CAT_CON_ORDEN),
    ('nom', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), CAT_NOM),
    ('te',  TargetEncoder(target_type='binary', random_state=RANDOM_STATE), CAT_TE + CAT_DERIVADAS),
    ('num', pipe_num(), NUM_TODAS),
], verbose_feature_names_out=False)

X_fe = nuevo_fe().fit_transform(X_train)
print("✅ Preprocesadores definidos.")
print(f"   One-hot (LR): {clone(preproc_lr).fit(X_fe, y_train).transform(X_fe.head()).shape[1]} columnas")
print(f"   Árboles: {clone(preproc_ordinal).fit(X_fe, y_train).transform(X_fe.head()).shape[1]} columnas")


---
# 🤖 8. Modelos de Machine Learning
**Protocolo idéntico para los 4 modelos:**
1. Búsqueda de hiperparámetros (`GridSearchCV`; `RandomizedSearchCV` en Random Forest y XGBoost, que tienen más combinaciones) con las mismas 5 particiones, optimizando ROC-AUC.
2. Probabilidades out-of-fold con la mejor configuración.
3. Reentrenamiento con todo el entrenamiento.

Las clases están balanceadas (≈50/50), así que no se usa SMOTE ni pesos de clase.

> ⏱️ Esta sección tarda más que en la versión anterior porque la búsqueda es más amplia (varias decenas de minutos en Colab gratuito). Para una corrida rápida de prueba baja `N_ITER_RF` y `N_ITER_XGB` en la celda siguiente.

In [ ]:
# ============================================================
# 8.0 FUNCIÓN COMÚN DE AJUSTE
# ============================================================
resultados_cv, modelos, prob_oof = {}, {}, {}

N_ITER_RF, N_ITER_XGB = 6, 10     # combinaciones que se prueban en Random Forest y XGBoost (súbelas para buscar más)

def ajustar_modelo(nombre, pipeline, rejilla, n_iter=None):
    if n_iter is None:
        busqueda = GridSearchCV(pipeline, rejilla, cv=CV, scoring='roc_auc', n_jobs=-1, refit=True)
    else:
        busqueda = RandomizedSearchCV(pipeline, rejilla, n_iter=n_iter, cv=CV, scoring='roc_auc',
                                      n_jobs=-1, refit=True, random_state=RANDOM_STATE)
    busqueda.fit(X_train, y_train)
    modelos[nombre] = busqueda.best_estimator_
    resultados_cv[nombre] = {
        'ROC-AUC CV': busqueda.best_score_,
        'desv.': busqueda.cv_results_['std_test_score'][busqueda.best_index_],
        'mejores parámetros': {k.replace('clf__', ''): v for k, v in busqueda.best_params_.items()},
    }
    prob_oof[nombre] = cross_val_predict(clone(busqueda.best_estimator_), X_train, y_train,
                                         cv=CV, method='predict_proba', n_jobs=-1)[:, 1]
    print(f"✅ {nombre}: ROC-AUC CV = {busqueda.best_score_:.4f} | {resultados_cv[nombre]['mejores parámetros']}")

base = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)

## 8.1 Regresión Logística (modelo de la entrega anterior)

In [ ]:
ajustar_modelo('Regresión Logística',
    Pipeline([('fe', nuevo_fe()), ('prep', clone(preproc_lr)),
              ('clf', LogisticRegression(max_iter=3000, random_state=RANDOM_STATE))]),
    {'clf__C': [0.03, 0.1, 0.3, 1]})

## 8.2 Decision Tree


In [ ]:
ajustar_modelo('Decision Tree',
    Pipeline([('fe', nuevo_fe()), ('prep', clone(preproc_ordinal)),
              ('clf', DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    {'clf__max_depth': [6, 8, 10, 14], 'clf__min_samples_leaf': [50, 100, 200]})

## 8.3 Random Forest


In [ ]:
ajustar_modelo('Random Forest',
    Pipeline([('fe', nuevo_fe()), ('prep', clone(preproc_ordinal)),
              ('clf', RandomForestClassifier(n_estimators=250, n_jobs=1, random_state=RANDOM_STATE))]),
    {'clf__max_depth': [14, 20, None], 'clf__min_samples_leaf': [10, 25, 50], 'clf__max_features': ['sqrt', 0.35]},
    n_iter=N_ITER_RF)

## 8.4 XGBoost
*Boosting*: construye árboles pequeños uno tras otro, y cada árbol nuevo corrige los errores de los anteriores.
Se ajustan la tasa de aprendizaje, el número de árboles, la profundidad y la regularización (`min_child_weight`, `reg_lambda`) para evitar el sobreajuste (`RandomizedSearchCV`).

In [ ]:
ajustar_modelo('XGBoost',
    Pipeline([('fe', nuevo_fe()), ('prep', clone(preproc_ordinal)),
              ('clf', XGBClassifier(subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                                    eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=1))]),
    {'clf__n_estimators': [300, 500, 800], 'clf__learning_rate': [0.03, 0.05, 0.08],
     'clf__max_depth': [3, 4, 5, 6], 'clf__min_child_weight': [5, 20, 50], 'clf__reg_lambda': [1, 10, 30]},
    n_iter=N_ITER_XGB)

## 8.5 Umbral y selección del modelo (solo con out-of-fold)
- **Umbral:** índice de Youden (máxima sensibilidad + especificidad). Con clases balanceadas equivale a minimizar **FP + FN**, que es lo que pide el profesor.
- **Regla de selección (fijada antes de ver la prueba):** gana el modelo con **menor FP + FN out-of-fold**. Si otro modelo está a menos del 0,5 % de diferencia, se prefiere el más simple.

In [ ]:
def errores(y_real, prob, umbral):
    return confusion_matrix(y_real, (prob >= umbral).astype(int), labels=[0, 1]).ravel()

umbrales, filas = {}, []
for nombre, p in prob_oof.items():
    fpr, tpr, thr = roc_curve(y_train, p)
    u = float(np.clip(thr, 0, 1)[np.argmax(tpr - fpr)])
    umbrales[nombre] = u
    for etiqueta, uu in [('0.5', 0.5), ('Youden', u)]:
        tn, fp, fn, tp = errores(y_train, p, uu)
        filas.append({'modelo': nombre, 'umbral': etiqueta, 'valor': uu, 'FP': fp, 'FN': fn,
                      'FP + FN': fp + fn, 'AUC OOF': roc_auc_score(y_train, p)})
tabla_oof = pd.DataFrame(filas).set_index(['modelo', 'umbral']).round(4)
display(tabla_oof)

SIMPLICIDAD = ['Regresión Logística', 'Decision Tree', 'Random Forest', 'XGBoost']
err_oof = tabla_oof.xs('Youden', level='umbral')['FP + FN']
mejor = err_oof.min()
candidatos = [m for m in SIMPLICIDAD if err_oof[m] <= mejor * 1.005]
MODELO_SELECCIONADO = candidatos[0]
print(f"\n🏆 Modelo seleccionado (validación): {MODELO_SELECCIONADO} | FP+FN OOF = {int(err_oof[MODELO_SELECCIONADO]):,} "
      f"| empatados dentro del 0,5 %: {candidatos}")

---
# 📊 9. Evaluación de los modelos (conjunto de prueba, uso único)

In [ ]:
prob_test = {n: m.predict_proba(X_test)[:, 1] for n, m in modelos.items()}
# Nombres de las columnas que ven los árboles (originales + derivadas)
COLUMNAS_ARBOL = list(modelos['Decision Tree'].named_steps['prep'].get_feature_names_out())

def metricas(y_real, prob, umbral):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
    return {'umbral': umbral, 'Accuracy': accuracy_score(y_real, pred),
            'Precision': precision_score(y_real, pred, zero_division=0),
            'Recall': recall_score(y_real, pred), 'Especificidad': tn / (tn + fp),
            'F1': f1_score(y_real, pred), 'ROC-AUC': roc_auc_score(y_real, prob),
            'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'FP + FN': fp + fn}

## 9.1 Matrices de confusión

In [ ]:
CLASES_TXT = ['Bajo (0)', 'Alto (1)']
fig, axes = plt.subplots(1, 5, figsize=(30, 6))
for ax, (nombre, p) in zip(axes, prob_test.items()):
    cm = confusion_matrix(y_test, (p >= umbrales[nombre]).astype(int), labels=[0, 1])
    txt = np.array([[f"TN\n{cm[0,0]:,}", f"FP\n{cm[0,1]:,}\n({cm[0,1]/cm.sum()*100:.1f}%)"],
                    [f"FN\n{cm[1,0]:,}\n({cm[1,0]/cm.sum()*100:.1f}%)", f"TP\n{cm[1,1]:,}"]])
    sns.heatmap(cm, annot=txt, fmt='', cmap='Blues', cbar=False, ax=ax, annot_kws={'fontsize': 12},
                xticklabels=CLASES_TXT, yticklabels=CLASES_TXT)
    marca = ' 🏆' if nombre == MODELO_SELECCIONADO else ''
    ax.set_title(f"{nombre}{marca}\numbral = {umbrales[nombre]:.3f}", fontweight='bold')
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')

# Último panel: matriz de la entrega anterior, reordenada con la convención actual (0 = bajo, 1 = alto)
cm_ant = np.array([[7978, 1726], [1679, 8060]])
txt = np.array([["TN\n7,978", "FP\n1,726\n(8.9%)"], ["FN\n1,679\n(8.6%)", "TP\n8,060"]])
sns.heatmap(cm_ant, annot=txt, fmt='', cmap='Greys', cbar=False, ax=axes[4], annot_kws={'fontsize': 12},
            xticklabels=CLASES_TXT, yticklabels=CLASES_TXT)
axes[4].set_title("Entrega anterior (referencia)\nRegresión Logística, umbral 0.5", fontweight='bold')
axes[4].set_xlabel('Predicción'); axes[4].set_ylabel('Real')
plt.suptitle(f'Matrices de confusión en prueba (n = {len(y_test):,})', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 9.2 Curvas ROC

In [ ]:
plt.figure(figsize=(8, 7))
for nombre, p in prob_test.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, lw=2.5 if nombre == MODELO_SELECCIONADO else 1.3,
             label=f"{nombre} (AUC = {roc_auc_score(y_test, p):.3f})")
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Azar')
plt.xlabel('Tasa de falsos positivos'); plt.ylabel('Sensibilidad (recall)')
plt.title('Curvas ROC en prueba', fontweight='bold'); plt.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 9.3 Métricas comparativas

In [ ]:
tabla_final = pd.DataFrame({n: metricas(y_test, p, umbrales[n]) for n, p in prob_test.items()}).T
tabla_final.loc['Entrega anterior (LR, 0.5)'] = {'umbral': 0.5, 'Accuracy': 0.8249,
    'Precision': 8060 / (8060 + 1726), 'Recall': 8060 / 9739, 'Especificidad': 7978 / 9704,
    'F1': 2 * 8060 / (2 * 8060 + 1726 + 1679), 'ROC-AUC': 0.9038,
    'TN': 7978, 'FP': 1726, 'FN': 1679, 'TP': 8060, 'FP + FN': 3405}
tabla_final['Reducción FP+FN vs anterior'] = (1 - tabla_final['FP + FN'] / 3405).map(lambda v: f"{v*100:.1f} %")
display(tabla_final.style.format({c: '{:.4f}' for c in ['umbral', 'Accuracy', 'Precision', 'Recall',
                                                        'Especificidad', 'F1', 'ROC-AUC']})
                         .format({c: '{:,.0f}' for c in ['TN', 'FP', 'FN', 'TP', 'FP + FN']}))
print(f"Línea base (clase mayoritaria) → accuracy {base.score(X_test, y_test):.3f}")

In [ ]:
# IC 95 % bootstrap del FP+FN y del AUC en prueba: ¿la mejora es real o es azar?
rng = np.random.default_rng(RANDOM_STATE)
y_np = y_test.to_numpy()
boot = {n: {'auc': [], 'err': []} for n in prob_test}
for _ in range(300):
    idx = rng.integers(0, len(y_np), len(y_np))
    for n, p in prob_test.items():
        boot[n]['auc'].append(roc_auc_score(y_np[idx], p[idx]))
        boot[n]['err'].append(((p[idx] >= umbrales[n]).astype(int) != y_np[idx]).sum())
display(pd.DataFrame({n: {'AUC IC95 inf': np.percentile(b['auc'], 2.5), 'AUC IC95 sup': np.percentile(b['auc'], 97.5),
                          'FP+FN IC95 inf': np.percentile(b['err'], 2.5), 'FP+FN IC95 sup': np.percentile(b['err'], 97.5)}
                      for n, b in boot.items()}).T.round(4))

## 9.4 Análisis adicional: zona de incertidumbre (clasificación con opción de rechazo)
Cuando la probabilidad está muy cerca del umbral, el modelo "duda". En vez de forzar una clase, esos estudiantes se marcan como **"revisar"**. Esto **reduce los FP y FN a costa de no clasificar a una parte de los estudiantes**, y por eso siempre se reporta la **cobertura** (% de estudiantes clasificados).

**Elección del ancho (con out-of-fold, no con la prueba):** la zona más amplia que mantiene una cobertura ≥ 85 %.

In [ ]:
# ============================================================
# 9.4 ZONA DE INCERTIDUMBRE
# ============================================================
p_oof_sel, u_sel = prob_oof[MODELO_SELECCIONADO], umbrales[MODELO_SELECCIONADO]
COBERTURA_MINIMA = 0.85

opciones = []
for ancho in np.arange(0.0, 0.31, 0.01):
    seguro = np.abs(p_oof_sel - u_sel) >= ancho
    tn, fp, fn, tp = errores(y_train[seguro], p_oof_sel[seguro], u_sel)
    opciones.append({'± ancho': round(ancho, 2), 'cobertura': seguro.mean(),
                     'error en clasificados': (fp + fn) / seguro.sum()})
opciones = pd.DataFrame(opciones)
ANCHO = opciones.loc[opciones['cobertura'] >= COBERTURA_MINIMA, '± ancho'].max()
display(opciones[opciones['± ancho'].isin([0, 0.05, 0.10, 0.15, 0.20, ANCHO])].round(3))
print(f"Zona elegida: probabilidad entre {u_sel - ANCHO:.2f} y {u_sel + ANCHO:.2f} → 'revisar'")

# Aplicación en prueba
p_te = prob_test[MODELO_SELECCIONADO]
seguro_te = np.abs(p_te - u_sel) >= ANCHO
cm_seg = confusion_matrix(y_test[seguro_te], (p_te[seguro_te] >= u_sel).astype(int), labels=[0, 1])
revisar = (~seguro_te).sum()

fig, ax = plt.subplots(figsize=(6, 5))
txt = np.array([[f"TN\n{cm_seg[0,0]:,}", f"FP\n{cm_seg[0,1]:,}"], [f"FN\n{cm_seg[1,0]:,}", f"TP\n{cm_seg[1,1]:,}"]])
sns.heatmap(cm_seg, annot=txt, fmt='', cmap='Greens', cbar=False, ax=ax, annot_kws={'fontsize': 13},
            xticklabels=['Alto (0)', 'Bajo (1)'], yticklabels=['Alto (0)', 'Bajo (1)'])
ax.set_title(f"{MODELO_SELECCIONADO} con zona de incertidumbre\n"
             f"Clasificados: {seguro_te.sum():,} ({seguro_te.mean()*100:.1f} %) | Revisar: {revisar:,}", fontweight='bold')
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
plt.tight_layout(); plt.show()

acc_seg = np.trace(cm_seg) / cm_seg.sum()
print(f"Accuracy en clasificados: {acc_seg:.4f} | FP = {cm_seg[0,1]:,} | FN = {cm_seg[1,0]:,}")
print(f"De los {revisar:,} marcados como 'revisar': "
      f"{(1 - y_test[~seguro_te].mean())*100:.1f} % eran de bajo rendimiento")

---
# 🔎 10. Interpretabilidad
La importancia **predictiva** no implica causalidad. Por ejemplo, el departamento puede reflejar diferencias de oferta educativa, conectividad o condiciones económicas regionales.

## 10.1 Coeficientes de la Regresión Logística

In [ ]:
lr = modelos['Regresión Logística']
nombres_lr = lr.named_steps['prep'].get_feature_names_out()
coef = pd.Series(lr.named_steps['clf'].coef_[0], index=[n.replace('cat__', '').replace('num__', '') for n in nombres_lr]).sort_values()
top = pd.concat([coef.head(12), coef.tail(12)])
plt.figure(figsize=(10, 9))
plt.barh(top.index, top.values, color=['#e74c3c' if v < 0 else '#2ecc71' for v in top.values])
plt.axvline(0, color='black', lw=0.8)
plt.title('Regresión Logística: coeficientes más extremos\n(verde = más probabilidad de alto rendimiento)', fontweight='bold')
plt.tight_layout(); plt.show()

## 10.2 Reglas del Decision Tree

In [ ]:
plt.figure(figsize=(26, 10))
plot_tree(modelos['Decision Tree'].named_steps['clf'], feature_names=COLUMNAS_ARBOL, class_names=['Bajo', 'Alto'],
          max_depth=3, filled=True, rounded=True, fontsize=9, proportion=True)
plt.title('Decision Tree (primeros 3 niveles)', fontweight='bold'); plt.show()
print("Códigos ordinales de las variables con orden (las variables con target encoding muestran su tasa de alto rendimiento):")
for col in CAT_CON_ORDEN:
    print(f"  {col}: " + ', '.join(f'{i}={c}' for i, c in enumerate(ORDENES[col])))

## 10.3 Importancia de variables (interna y permutation importance)

In [ ]:
# Importancia interna de los modelos de árboles (normalizada para que sume 1)
imp_interna = pd.DataFrame({n: pd.Series(modelos[n].named_steps['clf'].feature_importances_, index=COLUMNAS_ARBOL)
                            for n in ['Decision Tree', 'Random Forest', 'XGBoost']})
display(imp_interna.div(imp_interna.sum()).round(3).sort_values('XGBoost', ascending=False))

# Permutation importance en una muestra de prueba: se aplica a las 19 variables ORIGINALES,
# así que es comparable entre los 4 modelos aunque cada uno las codifique distinto.
muestra = X_test.sample(4000, random_state=RANDOM_STATE)
y_muestra = y_test.loc[muestra.index]
imp_perm = pd.DataFrame({n: pd.Series(permutation_importance(modelos[n], muestra, y_muestra, scoring='roc_auc',
                                                              n_repeats=5, random_state=RANDOM_STATE).importances_mean,
                                       index=PREDICTORAS) for n in modelos})
imp_perm = imp_perm.sort_values(MODELO_SELECCIONADO)
imp_perm.plot.barh(figsize=(10, 9))
plt.xlabel('Caída del ROC-AUC al permutar la variable')
plt.title('Permutation importance (muestra de prueba)', fontweight='bold')
plt.tight_layout(); plt.show()

## 10.4 SHAP (XGBoost)

In [ ]:
try:
    import shap
    pipe = modelos['XGBoost']
    Xs = pd.DataFrame(Pipeline(pipe.steps[:-1]).transform(X_test.sample(1500, random_state=RANDOM_STATE)),
                      columns=COLUMNAS_ARBOL)
    valores = shap.TreeExplainer(pipe.named_steps['clf']).shap_values(Xs)
    shap.summary_plot(valores, Xs, show=True)
except ImportError:
    print("SHAP no está instalado. Ejecuta: !pip install shap")

---
# 🧪 11. Validación temporal complementaria
Se entrena con **2018–2021** y se evalúa con **2022**, con los hiperparámetros ya elegidos y el umbral recalculado con out-of-fold dentro de 2018–2021. Mide si el patrón se mantiene en un año que el modelo no vio.

In [ ]:
es_2022 = (anio_modelo == 2022)
X_a, y_a, X_b, y_b = X[~es_2022], y[~es_2022], X[es_2022], y[es_2022]
modelo_t = MODELO_SELECCIONADO

p_oof_t = cross_val_predict(clone(modelos[modelo_t]), X_a, y_a, cv=CV, method='predict_proba', n_jobs=-1)[:, 1]
fpr, tpr, thr = roc_curve(y_a, p_oof_t)
u_t = float(np.clip(thr, 0, 1)[np.argmax(tpr - fpr)])
m_t = clone(modelos[modelo_t]).fit(X_a, y_a)
res_t = metricas(y_b, m_t.predict_proba(X_b)[:, 1], u_t)
print(f"Modelo: {modelo_t} | entrenamiento 2018–2021: {len(y_a):,} | prueba 2022: {len(y_b):,}")
display(pd.DataFrame({'División aleatoria': tabla_final.loc[modelo_t, ['ROC-AUC', 'Accuracy', 'Recall', 'Especificidad']],
                      'Temporal (2022)': pd.Series(res_t)[['ROC-AUC', 'Accuracy', 'Recall', 'Especificidad']]}).astype(float).round(4))

---
# ✅ 12. Conclusiones
La celda siguiente imprime el resumen con los valores de esta ejecución.

In [ ]:
sel = tabla_final.loc[MODELO_SELECCIONADO]
print(f"🏆 Modelo seleccionado: {MODELO_SELECCIONADO} (umbral {sel['umbral']:.3f}, elegido con validación)")
print(f"   Prueba → Accuracy {sel['Accuracy']:.4f} | Recall {sel['Recall']:.4f} | Especificidad {sel['Especificidad']:.4f} | "
      f"F1 {sel['F1']:.4f} | ROC-AUC {sel['ROC-AUC']:.4f}")
print(f"   FP = {int(sel['FP']):,} | FN = {int(sel['FN']):,} | FP+FN = {int(sel['FP + FN']):,} "
      f"(entrega anterior: 3,405 → reducción {(1 - sel['FP + FN']/3405)*100:.1f} %)")
print(f"\n   Con zona de incertidumbre: cobertura {seguro_te.mean()*100:.1f} % | FP = {cm_seg[0,1]:,} | FN = {cm_seg[1,0]:,} | "
      f"accuracy {acc_seg:.4f} (reducción FP+FN vs anterior: {(1 - (cm_seg[0,1]+cm_seg[1,0])/3405)*100:.1f} %)")

**Conclusiones metodológicas**
1. **Datos:** se corrigió la codificación del texto, se eliminaron la columna constante, el identificador y las columnas derivadas del puntaje, y se excluyeron los estudiantes con promedio 0 (módulos sin calificación según el ICFES). Los faltantes se trataron como una categoría propia.
2. **Comparación justa:** los 4 modelos usaron las mismas particiones, la misma métrica de ajuste y el mismo criterio de umbral. El modelo ganador se eligió con validación; la prueba se usó una sola vez.
3. **Variables derivadas:** se añadieron resúmenes (bienes del hogar, educación de los padres, fuentes de pago, rango de edad) y cruces (departamento × origen, carácter × origen, origen × metodología, origen × matrícula, departamento × estrato) dentro del Pipeline, sin usar el puntaje. Su aporte se mide en la tabla 9.3 frente a la entrega anterior.
4. **Reducción de errores:** los modelos de conjunto de árboles (Random Forest y, sobre todo, XGBoost) capturan interacciones entre variables (por ejemplo, departamento × tipo de institución) que el modelo lineal no ve, y reducen FP y FN frente a la Regresión Logística de la entrega anterior. El Decision Tree, más simple, sacrifica desempeño a cambio de reglas legibles. El intervalo bootstrap indica si la mejora es estadísticamente consistente.
5. **Zona de incertidumbre:** al no forzar la clasificación de los casos dudosos, FP y FN bajan de forma importante. Los estudiantes marcados como "revisar" son justamente los que necesitan una evaluación individual.
6. **Variables más importantes:** las institucionales (carácter, metodología, origen), la región y la edad pesan tanto o más que las familiares.
7. **Limitaciones:** (a) el archivo no trae los 5 módulos, así que no se pudieron excluir los estudiantes con un solo módulo en 0; (b) las importancias son asociaciones, no causas; (c) el modelo distingue los extremos (P20 vs P80), no a cualquier estudiante; (d) parte del error es irreducible: estudiantes con el mismo perfil tienen resultados distintos.

---
# 🖥️ 13. Aplicación interactiva (Streamlit)
Esta sección genera todos los archivos de la aplicación:

| Archivo | Contenido |
|---|---|
| `app.py` | La aplicación Streamlit |
| `features_saber_pro.py` | Variables derivadas (los modelos guardados lo necesitan para cargarse) |
| `modelos_saber_pro.joblib` | Regresión Logística, Decision Tree, Random Forest y el preprocesador de XGBoost |
| `xgboost_modelo.json` | El modelo XGBoost en su formato nativo (portable entre Windows, Mac y Linux) |
| `resultados_saber_pro.joblib` | Umbrales, métricas, probabilidades de prueba, importancias y opciones de cada variable |
| `requirements.txt` / `packages.txt` | Dependencias, para publicarla en Streamlit Community Cloud |
| `README.md` | Instrucciones |

**Qué incluye la app:** predicción individual con las 19 variables, comparación de los 4 modelos, análisis "¿qué pasaría si…?", matriz de confusión y métricas de cada modelo, curvas ROC, gráfica de errores (FP y FN) por modelo e importancia de variables.

## 13.1 Guardar modelos y resultados

In [ ]:
# ============================================================
# 13.1 EXPORTAR MODELOS Y RESULTADOS
# ============================================================
import joblib

ETIQUETAS_VAR = {
    'ESTU_DEPTO_RESIDE': ('Departamento de residencia', 'Estudiante'),
    'ESTU_GENERO': ('Género', 'Estudiante'),
    'EDAD': ('Edad', 'Estudiante'),
    'ESTU_HORASSEMANATRABAJA': ('Horas de trabajo semanales', 'Estudiante'),
    'FAMI_ESTRATOVIVIENDA': ('Estrato de la vivienda', 'Hogar y familia'),
    'FAMI_EDUCACIONPADRE': ('Educación del padre', 'Hogar y familia'),
    'FAMI_EDUCACIONMADRE': ('Educación de la madre', 'Hogar y familia'),
    'FAMI_TIENEINTERNET': ('Internet en el hogar', 'Hogar y familia'),
    'FAMI_TIENECOMPUTADOR': ('Computador en el hogar', 'Hogar y familia'),
    'FAMI_TIENEAUTOMOVIL': ('Automóvil en el hogar', 'Hogar y familia'),
    'FAMI_TIENELAVADORA': ('Lavadora en el hogar', 'Hogar y familia'),
    'ESTU_VALORMATRICULAUNIVERSIDAD': ('Valor de la matrícula', 'Financiación'),
    'ESTU_PAGOMATRICULABECA': ('Paga con beca', 'Financiación'),
    'ESTU_PAGOMATRICULACREDITO': ('Paga con crédito', 'Financiación'),
    'ESTU_PAGOMATRICULAPADRES': ('Pagan los padres', 'Financiación'),
    'ESTU_PAGOMATRICULAPROPIO': ('Paga con recursos propios', 'Financiación'),
    'INST_CARACTER_ACADEMICO': ('Carácter de la institución', 'Institución'),
    'INST_ORIGEN': ('Origen de la institución', 'Institución'),
    'ESTU_METODO_PRGM': ('Metodología del programa', 'Institución'),
}

variables = []
for col in PREDICTORAS:
    etiqueta, grupo = ETIQUETAS_VAR[col]
    if col in NUMERICAS:
        variables.append({'col': col, 'etiqueta': etiqueta, 'grupo': grupo, 'tipo': 'numerica',
                          'min': int(X_train[col].min()), 'max': int(X_train[col].max()),
                          'defecto': int(X_train[col].median())})
    else:
        presentes = set(X_train[col].unique())
        if col in ORDENES:
            opciones = [o for o in ORDENES[col] if o in presentes]
        else:
            opciones = sorted(presentes - {CAT_FALTANTE}) + ([CAT_FALTANTE] if CAT_FALTANTE in presentes else [])
        variables.append({'col': col, 'etiqueta': etiqueta, 'grupo': grupo, 'tipo': 'categorica',
                          'opciones': opciones, 'defecto': X_train[col].mode()[0]})

metricas_app = tabla_final.drop(index='Entrega anterior (LR, 0.5)').drop(columns=['Reducción FP+FN vs anterior'])
metricas_app = metricas_app.astype(float)

resultados_app = {
    'variables': variables,
    'predictoras': PREDICTORAS,
    'umbrales': {n: float(u) for n, u in umbrales.items()},
    'ancho': float(ANCHO),
    'seleccionado': MODELO_SELECCIONADO,
    'metricas': metricas_app,
    'y_test': y_test.to_numpy(),
    'prob_test': {n: np.asarray(p) for n, p in prob_test.items()},
    'importancias': imp_perm,
    'cortes': dataset.groupby(COL_ANIO)[['CORTE_BAJO', 'CORTE_ALTO']].first().round(1),
    'anterior': {'TN': 7978, 'FP': 1726, 'FN': 1679, 'TP': 8060},
    'n_total': int(len(X)), 'n_train': int(len(X_train)), 'n_test': int(len(X_test)),
}

# XGBoost se guarda en su formato nativo JSON: un pickle de XGBoost puede no abrirse en otro sistema
# operativo (p. ej., guardado en Colab/Linux y abierto en Windows). Su preprocesador sí va en el joblib.
modelos[ 'XGBoost'].named_steps['clf'].save_model('xgboost_modelo.json')
modelos_export = {n: ({'tipo': 'xgboost_json', 'archivo': 'xgboost_modelo.json',
                       'prep': Pipeline(m.steps[:-1])} if n == 'XGBoost' else m)
                  for n, m in modelos.items()}
joblib.dump(modelos_export, 'modelos_saber_pro.joblib', compress=3)
joblib.dump(resultados_app, 'resultados_saber_pro.joblib', compress=3)

def reconstruir(modelos_cargados):
    # Arma de nuevo el Pipeline de XGBoost a partir del preprocesador + el archivo JSON
    salida = {}
    for n, m in modelos_cargados.items():
        if isinstance(m, dict) and m.get('tipo') == 'xgboost_json':
            clf = XGBClassifier()
            clf.load_model(m['archivo'])
            m = Pipeline([('prep', m['prep']), ('clf', clf)])
        salida[n] = m
    return salida

# Verificación: se vuelven a cargar (igual que en la app) y se comparan las predicciones
m_chk = reconstruir(joblib.load('modelos_saber_pro.joblib'))
r_chk = joblib.load('resultados_saber_pro.joblib')
dif = np.abs(m_chk['XGBoost'].predict_proba(X_test)[:, 1] - modelos['XGBoost'].predict_proba(X_test)[:, 1]).max()
print(f"Diferencia máxima XGBoost original vs recargado: {dif:.2e}")
ejemplo = pd.DataFrame([{v['col']: v['defecto'] for v in r_chk['variables']}])[PREDICTORAS]
print("✅ Modelos guardados y verificados. P(alto) del perfil más frecuente:")
for n, m in m_chk.items():
    print(f"   {n:20s} {m.predict_proba(ejemplo)[0, 1]:.3f}")
import os
for f in ['features_saber_pro.py', 'modelos_saber_pro.joblib', 'resultados_saber_pro.joblib', 'xgboost_modelo.json']:
    print(f"   {f}: {os.path.getsize(f) / 1024**2:.1f} MB")

## 13.2 Crear `app.py`

In [ ]:
%%writefile app.py
# ============================================================
# 🎓 Saber Pro · Predictor de rendimiento académico (Streamlit)
# ============================================================
# Ejecutar:  streamlit run app.py
# Necesita en la misma carpeta: features_saber_pro.py, modelos_saber_pro.joblib, xgboost_modelo.json y resultados_saber_pro.joblib
import numpy as np
import pandas as pd
import joblib
import streamlit as st
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix, roc_curve
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

st.set_page_config(page_title="Saber Pro · Predictor de rendimiento", page_icon="🎓",
                   layout="wide", initial_sidebar_state="expanded")

try:
    import features_saber_pro  # noqa: F401  (los Pipelines guardados lo referencian por nombre)
except ImportError:
    st.error("Falta el archivo `features_saber_pro.py`. Cópialo en la misma carpeta que `app.py`.")
    st.stop()

AZUL, VERDE, ROJO, AMBAR = "#1f4e79", "#1e9e6a", "#d64545", "#e0a100"
COLORES_MODELO = ["#1f4e79", "#e07b39", "#2a9d8f", "#8e44ad"]

st.markdown("""
<style>
.block-container {padding-top: 1.5rem; padding-bottom: 2rem;}
.hero {background: linear-gradient(120deg, #1f4e79 0%, #2f7fb8 55%, #3fa9d6 100%); color: white;
       padding: 1.4rem 1.8rem; border-radius: 18px; margin-bottom: 1.1rem;
       box-shadow: 0 10px 28px rgba(31,78,121,.25);}
.hero h1 {color: white; font-size: 1.85rem; margin: 0; font-weight: 800;}
.hero p {color: #e6f1fb; margin: .35rem 0 0 0; font-size: 1rem;}
.chip {display: inline-block; background: rgba(255,255,255,.18); color: white; padding: .18rem .7rem;
       border-radius: 999px; font-size: .78rem; margin: .55rem .35rem 0 0;}
.result {border-radius: 18px; padding: 1.4rem 1.5rem; color: white; text-align: center;
         box-shadow: 0 8px 22px rgba(0,0,0,.12);}
.result h2 {color: white; margin: 0; font-size: 1.75rem; font-weight: 800;}
.result p {margin: .35rem 0 0 0; color: rgba(255,255,255,.93); font-size: .98rem;}
.nota {background: #f3f6fa; border-left: 4px solid #2f7fb8; padding: .75rem 1rem; border-radius: 8px;
       color: #374151; font-size: .9rem;}
div[data-testid="stMetric"] {background: rgba(47,127,184,.06); border: 1px solid rgba(47,127,184,.18);
       border-radius: 14px; padding: .7rem .9rem;}
.stTabs [data-baseweb="tab-list"] {gap: .35rem;}
.stTabs [data-baseweb="tab"] {background: rgba(47,127,184,.08); border-radius: 10px 10px 0 0; padding: .5rem 1rem;}
.stTabs [aria-selected="true"] {background: #1f4e79 !important; color: white !important;}
.foot {color: #9ca3af; font-size: .8rem; text-align: center; margin-top: 2.2rem;}
</style>
""", unsafe_allow_html=True)


# ------------------------------------------------------------
# Carga de modelos y resultados
# ------------------------------------------------------------
@st.cache_resource(show_spinner="Cargando modelos…")
def cargar():
    modelos = {}
    for n, m in joblib.load("modelos_saber_pro.joblib").items():
        # XGBoost viene en formato JSON nativo (portable entre sistemas operativos)
        if isinstance(m, dict) and m.get("tipo") == "xgboost_json":
            clf = XGBClassifier()
            clf.load_model(m["archivo"])
            m = Pipeline([("prep", m["prep"]), ("clf", clf)])
        modelos[n] = m
    return modelos, joblib.load("resultados_saber_pro.joblib")


try:
    MODELOS, R = cargar()
except FileNotFoundError:
    st.error("No se encontraron los archivos de modelos. Copia `modelos_saber_pro.joblib`, "
             "`xgboost_modelo.json`, `resultados_saber_pro.joblib` y `features_saber_pro.py` en la misma carpeta que `app.py`.")
    st.stop()

VARS = R["variables"]
PREDICTORAS = R["predictoras"]
NOMBRES = list(MODELOS)
ETIQ = {v["col"]: v["etiqueta"] for v in VARS}
FORMATO = {"F": "Femenino", "M": "Masculino"}
GRUPOS = [("👤", "Estudiante"), ("🏠", "Hogar y familia"), ("💳", "Financiación"), ("🏛️", "Institución")]
Y_TEST = np.asarray(R["y_test"])


def fmt(x):
    return FORMATO.get(x, x)


def clasificar(p, u):
    return "alto" if p >= u else "bajo"


def fig_matriz(cm, titulo, escala="Blues"):
    total = cm.sum()
    nombres = [["TN", "FP"], ["FN", "TP"]]
    texto = [[f"<b>{nombres[i][j]}</b><br>{cm[i, j]:,}<br>({cm[i, j] / total * 100:.1f} %)" for j in range(2)]
             for i in range(2)]
    fig = go.Figure(go.Heatmap(z=cm, x=["Bajo (0)", "Alto (1)"], y=["Bajo (0)", "Alto (1)"], text=texto,
                               texttemplate="%{text}", textfont={"size": 15}, colorscale=escala,
                               showscale=False, hoverinfo="skip"))
    fig.update_layout(title=titulo, height=360, margin=dict(l=10, r=10, t=50, b=10),
                      xaxis_title="Predicción", yaxis_title="Real", yaxis_autorange="reversed")
    return fig


# ------------------------------------------------------------
# Barra lateral
# ------------------------------------------------------------
with st.sidebar:
    st.markdown("## 🎓 Saber Pro ML")
    st.caption("Clasificación de rendimiento académico (alto vs. bajo) a partir de 19 variables "
               "sociodemográficas, socioeconómicas, de financiación e institucionales.")
    modelo = st.selectbox("Modelo", NOMBRES, index=NOMBRES.index(R["seleccionado"]),
                          format_func=lambda n: f"{n}  🏆" if n == R["seleccionado"] else n)
    umbral = float(R["umbrales"][modelo])   # punto de corte elegido con validación cruzada
    st.divider()
    st.markdown("**Convención**  \n🔴 0 = Bajo rendimiento  \n🟢 1 = Alto rendimiento")
    st.caption("Elige el modelo con el que quieres hacer las predicciones y ver su desempeño. "
               "🏆 marca el modelo con mejor desempeño en validación.")


st.markdown(f"""
<div class="hero">
  <h1>🎓 Predictor de rendimiento en Saber Pro</h1>
  <p>Explora cómo se asocian las características de un estudiante con su probabilidad de pertenecer al
     grupo de alto rendimiento (≥ P80 de su año) o bajo rendimiento (≤ P20).</p>
  <span class="chip">📚 {R['n_total']:,} estudiantes · 2018–2022</span>
  <span class="chip">🤖 {len(NOMBRES)} modelos</span>
  <span class="chip">🏆 Mejor: {R['seleccionado']}</span>
  <span class="chip">🧪 Prueba: {R['n_test']:,} estudiantes</span>
</div>""", unsafe_allow_html=True)

tab_pred, tab_eval, tab_imp, tab_info = st.tabs(
    ["🔮 Predicción individual", "📊 Desempeño de modelos", "🔎 Importancia de variables",
     "ℹ️ Acerca del proyecto"])

# ============================================================
# 1. PREDICCIÓN INDIVIDUAL
# ============================================================
with tab_pred:
    st.markdown("#### Perfil del estudiante")
    with st.form("formulario_perfil"):
        valores = {}
        for icono, grupo in GRUPOS:
            vars_g = [v for v in VARS if v["grupo"] == grupo]
            st.markdown(f"**{icono} {grupo}**")
            cols = st.columns(3)
            for i, v in enumerate(vars_g):
                with cols[i % 3]:
                    if v["tipo"] == "numerica":
                        valores[v["col"]] = st.number_input(v["etiqueta"], int(v["min"]), int(v["max"]),
                                                            int(v["defecto"]), 1)
                    else:
                        valores[v["col"]] = st.selectbox(v["etiqueta"], v["opciones"],
                                                         index=v["opciones"].index(v["defecto"]), format_func=fmt)
        enviado = st.form_submit_button("🔮 Calcular predicción", type="primary", use_container_width=True)

    if enviado:
        st.session_state["perfil_estudiante"] = valores

    if "perfil_estudiante" not in st.session_state:
        st.info("Completa el perfil y pulsa **Calcular predicción**. Los valores iniciales corresponden "
                "a la categoría más frecuente de cada variable.")
    else:
        perfil = pd.DataFrame([st.session_state["perfil_estudiante"]])[PREDICTORAS]
        probs = {n: float(m.predict_proba(perfil)[:, 1][0]) for n, m in MODELOS.items()}
        p = probs[modelo]
        clase = clasificar(p, umbral)
        estilo = {"alto": (VERDE, "🟢 Alto rendimiento", "El perfil se parece más al grupo ≥ P80."),
                  "bajo": (ROJO, "🔴 Bajo rendimiento", "El perfil se parece más al grupo ≤ P20.")}[clase]

        c1, c2 = st.columns([1, 1.15])
        with c1:
            st.markdown(f"""<div class="result" style="background:{estilo[0]}">
                <h2>{estilo[1]}</h2><p>{estilo[2]}</p>
                <p style="margin-top:.8rem;font-size:1.05rem"><b>{modelo}</b> · Probabilidad de alto rendimiento:
                <b>{p*100:.1f} %</b></p></div>""", unsafe_allow_html=True)
            st.write("")
            comp = pd.DataFrame({"Modelo": list(probs), "P(alto)": [probs[n] for n in probs]})
            comp["Clasificación"] = [
                ("🟢 Alto" if clasificar(probs[n], R["umbrales"][n]) == "alto" else "🔴 Bajo") for n in probs]
            st.dataframe(comp.style.format({"P(alto)": "{:.1%}"}), hide_index=True, use_container_width=True)
        with c2:
            pasos = [{"range": [0, umbral * 100], "color": "#fbe3e3"},
                     {"range": [umbral * 100, 100], "color": "#dff5ea"}]
            g = go.Figure(go.Indicator(mode="gauge+number", value=p * 100, number={"suffix": " %"},
                                       title={"text": "Probabilidad de alto rendimiento"},
                                       gauge={"axis": {"range": [0, 100]}, "bar": {"color": estilo[0]},
                                              "steps": pasos,
                                              "threshold": {"line": {"color": "#111827", "width": 4},
                                                            "value": umbral * 100}}))
            g.update_layout(height=300, margin=dict(l=20, r=20, t=60, b=10))
            st.plotly_chart(g, use_container_width=True)

        # Barras comparando los 4 modelos
        fb = go.Figure()
        for i, n in enumerate(probs):
            fb.add_bar(y=[n], x=[probs[n] * 100], orientation="h", marker_color=COLORES_MODELO[i % 4],
                       text=f"{probs[n]*100:.1f} %", textposition="outside", showlegend=False)
        fb.add_vline(x=50, line_dash="dot", line_color="#9ca3af")
        fb.update_layout(title="¿Qué dice cada modelo?", xaxis_title="P(alto) %", xaxis_range=[0, 105],
                         height=260, margin=dict(l=10, r=10, t=50, b=10))
        st.plotly_chart(fb, use_container_width=True)

        # Análisis "¿qué pasaría si…?"
        st.markdown("#### 🧭 ¿Qué cambios moverían más la probabilidad?")
        filas, etiquetas = [], []
        for v in VARS:
            actual = st.session_state["perfil_estudiante"][v["col"]]
            alternativas = (list(range(int(v["min"]), int(v["max"]) + 1, 2)) if v["tipo"] == "numerica"
                            else [o for o in v["opciones"] if o not in (actual, "Sin información")])
            for alt in alternativas:
                fila = dict(st.session_state["perfil_estudiante"])
                fila[v["col"]] = alt
                filas.append(fila)
                etiquetas.append((v["etiqueta"], alt))
        if filas:
            p_alt = MODELOS[modelo].predict_proba(pd.DataFrame(filas)[PREDICTORAS])[:, 1]
            efectos = pd.DataFrame(etiquetas, columns=["variable", "valor"])
            efectos["cambio"] = (p_alt - p) * 100
            mejor = efectos.loc[efectos.groupby("variable")["cambio"].apply(lambda s: s.abs().idxmax())]
            mejor = mejor.reindex(mejor["cambio"].abs().sort_values(ascending=False).index).head(8)
            mejor["texto"] = [f"{r.variable} → {fmt(r.valor)}" for r in mejor.itertuples()]
            fw = go.Figure(go.Bar(y=mejor["texto"][::-1], x=mejor["cambio"][::-1], orientation="h",
                                  marker_color=[VERDE if c > 0 else ROJO for c in mejor["cambio"][::-1]],
                                  text=[f"{c:+.1f} pp" for c in mejor["cambio"][::-1]], textposition="outside"))
            fw.update_layout(height=380, margin=dict(l=10, r=40, t=10, b=10),
                             xaxis_title="Cambio en P(alto), puntos porcentuales")
            st.plotly_chart(fw, use_container_width=True)
        st.markdown('<div class="nota">⚠️ Estos cambios muestran <b>asociaciones aprendidas por el modelo</b>, '
                    'no efectos causales: cambiar una característica en la vida real no garantiza el mismo '
                    'cambio en el rendimiento.</div>', unsafe_allow_html=True)

# ============================================================
# 2. DESEMPEÑO DE MODELOS
# ============================================================
with tab_eval:
    p_test = np.asarray(R["prob_test"][modelo])
    cm = confusion_matrix(Y_TEST, (p_test >= umbral).astype(int), labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    n_clas = cm.sum()

    st.markdown(f"#### {modelo} en el conjunto de prueba ({n_clas:,} estudiantes)")
    k = st.columns(6)
    k[0].metric("Accuracy", f"{(tp + tn) / n_clas:.3f}")
    k[1].metric("Recall (alto)", f"{tp / (tp + fn):.3f}")
    k[2].metric("Especificidad (bajo)", f"{tn / (tn + fp):.3f}")
    k[3].metric("ROC-AUC", f"{R['metricas'].loc[modelo, 'ROC-AUC']:.3f}")
    k[4].metric("Falsos positivos (FP)", f"{fp:,}")
    k[5].metric("Falsos negativos (FN)", f"{fn:,}")

    c1, c2 = st.columns(2)
    with c1:
        st.plotly_chart(fig_matriz(cm, f"Matriz de confusión · {modelo}"), use_container_width=True)
        st.markdown('<div class="nota"><b>FP</b>: el modelo dijo <i>alto</i> pero era de bajo rendimiento '
                    '(estudiante en riesgo no detectado).<br><b>FN</b>: el modelo dijo <i>bajo</i> pero era de '
                    'alto rendimiento.</div>', unsafe_allow_html=True)
    with c2:
        fr = go.Figure()
        for i, n in enumerate(NOMBRES):
            pn = np.asarray(R["prob_test"][n])
            f_, t_, _ = roc_curve(Y_TEST, pn)
            auc = R["metricas"].loc[n, "ROC-AUC"]
            fr.add_scatter(x=f_, y=t_, name=f"{n} (AUC {auc:.3f})",
                           line=dict(color=COLORES_MODELO[i % 4], width=3.5 if n == modelo else 1.6))
        fr.add_scatter(x=[0, 1], y=[0, 1], name="Azar", line=dict(color="gray", dash="dash"))
        fr.update_layout(title="Curvas ROC (los 4 modelos)", xaxis_title="Tasa de falsos positivos",
                         yaxis_title="Sensibilidad", height=420, margin=dict(l=10, r=10, t=50, b=10),
                         legend=dict(orientation="h", y=-0.25))
        st.plotly_chart(fr, use_container_width=True)

    st.markdown("#### Comparación de los 4 modelos")
    fe = go.Figure()
    for col, color in [("FP", ROJO), ("FN", AZUL)]:
        fe.add_bar(x=NOMBRES, y=[R["metricas"].loc[n, col] for n in NOMBRES], name=col, marker_color=color,
                   text=[f"{int(R['metricas'].loc[n, col]):,}" for n in NOMBRES], textposition="outside")
    fe.update_layout(barmode="group", title="Errores por modelo", yaxis_title="Estudiantes", height=380,
                     margin=dict(l=10, r=10, t=50, b=10), legend=dict(orientation="h", y=-0.15))
    st.plotly_chart(fe, use_container_width=True)
    tabla = R["metricas"].drop(columns=["umbral"], errors="ignore").copy()
    cols_num = ["Accuracy", "Precision", "Recall", "Especificidad", "F1", "ROC-AUC"]
    st.dataframe(tabla.style.format({c: "{:.4f}" for c in cols_num if c in tabla})
                 .format({c: "{:,.0f}" for c in ["TN", "FP", "FN", "TP", "FP + FN"] if c in tabla})
                 .highlight_min(subset=["FP + FN"], color="#d1fae5"),
                 use_container_width=True)

# ============================================================
# 3. IMPORTANCIA DE VARIABLES
# ============================================================
with tab_imp:
    st.markdown("#### Permutation importance")
    st.caption("Cuánto cae el ROC-AUC en prueba cuando se desordena cada variable. Cuanto mayor la caída, "
               "más depende el modelo de esa variable para distinguir los grupos.")
    elegidos = st.multiselect("Modelos a comparar", NOMBRES, default=[modelo])
    imp = R["importancias"].copy()
    if elegidos:
        orden = imp[elegidos].mean(axis=1).sort_values().index
        fi = go.Figure()
        for i, n in enumerate(elegidos):
            fi.add_bar(y=[ETIQ.get(c, c) for c in orden], x=imp.loc[orden, n], name=n, orientation="h",
                       marker_color=COLORES_MODELO[NOMBRES.index(n) % 4])
        fi.update_layout(barmode="group", height=640, margin=dict(l=10, r=10, t=10, b=10),
                         xaxis_title="Caída del ROC-AUC", legend=dict(orientation="h", y=-0.08))
        st.plotly_chart(fi, use_container_width=True)
    st.markdown('<div class="nota">La importancia es <b>predictiva</b>, no causal. Variables asociadas entre sí '
                '(p. ej., estrato y automóvil) se reparten la información, así que su importancia individual '
                'puede verse menor.</div>', unsafe_allow_html=True)

# ============================================================
# 4. ACERCA DEL PROYECTO
# ============================================================
with tab_info:
    st.markdown(f"""
#### Metodología
- **Datos:** resultados Saber Pro 2018–2022 (muestra del 20 %), {R['n_total']:,} estudiantes en los extremos de la distribución.
- **Variable objetivo:** por año, **1 = alto** (puntaje promedio ≥ P80) y **0 = bajo** (≤ P20). Se excluye la zona intermedia.
- **Predictores:** 19 variables sociodemográficas, socioeconómicas, laborales, de financiación e institucionales.
  El puntaje y todo lo derivado de él **no** se usan como predictores. Con esas 19 respuestas el modelo calcula internamente variables derivadas
  (bienes del hogar, educación máxima de los padres, rango de edad y cruces como departamento × origen de la institución).
- **Validación:** división estratificada 80/20 ({R['n_train']:,} / {R['n_test']:,}); hiperparámetros con validación cruzada de 5 particiones;
  punto de corte de cada modelo elegido con validación cruzada, sin usar el conjunto de prueba.
- **Modelos (uno por familia):** Regresión Logística (lineal; modelo de la entrega anterior), Decision Tree (árbol individual),
  Random Forest (*bagging*) y XGBoost (*boosting*).

#### Limitaciones
- Las importancias y los análisis "¿qué pasaría si…?" muestran **asociaciones**, no causas.
- El modelo distingue los extremos (P20 vs P80); no está pensado para clasificar a estudiantes del grupo intermedio.
- El archivo original solo trae el promedio, así que no se pudieron detectar los estudiantes con un único módulo en 0.
""")
    st.markdown("#### Cortes de la variable objetivo por año")
    st.dataframe(R["cortes"], use_container_width=True)

st.markdown('<div class="foot">Proyecto académico · Introducción a los Sistemas Inteligentes · UAJS</div>',
            unsafe_allow_html=True)

## 13.3 Dependencias e instrucciones

In [ ]:
# ============================================================
# 13.3 requirements.txt, packages.txt y README.md
# ============================================================
import sys, sklearn, xgboost, plotly

# Se fijan las MISMAS versiones de Colab: los modelos guardados solo cargan bien con ellas
requisitos = f"""streamlit==1.40.0
scikit-learn=={sklearn.__version__}
xgboost=={xgboost.__version__}
pandas=={pd.__version__}
numpy=={np.__version__}
joblib=={joblib.__version__}
plotly=={plotly.__version__}
"""
open('requirements.txt', 'w').write(requisitos)
open('packages.txt', 'w').write('libgomp1\n')   # librería del sistema (OpenMP) que usa XGBoost

readme = f"""# Saber Pro · Predictor de rendimiento (Streamlit)

Archivos necesarios (todos en la misma carpeta):
app.py, features_saber_pro.py, modelos_saber_pro.joblib, xgboost_modelo.json, resultados_saber_pro.joblib, requirements.txt, packages.txt

## Ejecutar en un computador
1. Instalar Python {sys.version_info.major}.{sys.version_info.minor}
2. pip install -r requirements.txt
3. streamlit run app.py

## Publicar en Streamlit Community Cloud
1. Subir todos los archivos a un repositorio de GitHub.
2. En https://share.streamlit.io → Create app → elegir el repositorio, rama main y archivo app.py.
3. En "Advanced settings" elegir Python {sys.version_info.major}.{sys.version_info.minor}.
"""
open('README.md', 'w').write(readme)
print(requisitos)
print(f"Python de este entorno: {sys.version_info.major}.{sys.version_info.minor}")

## 13.4 Descargar los archivos

In [ ]:
# ============================================================
# 13.4 EMPAQUETAR Y DESCARGAR
# ============================================================
import zipfile
ARCHIVOS_APP = ['app.py', 'features_saber_pro.py', 'modelos_saber_pro.joblib', 'xgboost_modelo.json', 'resultados_saber_pro.joblib',
                'requirements.txt', 'packages.txt', 'README.md']
with zipfile.ZipFile('app_saber_pro.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ARCHIVOS_APP:
        z.write(f)
print(f"✅ app_saber_pro.zip creado ({os.path.getsize('app_saber_pro.zip') / 1024**2:.1f} MB) con: {ARCHIVOS_APP}")

try:
    from google.colab import files
    files.download('app_saber_pro.zip')
except ImportError:
    print("Fuera de Colab: el archivo quedó en la carpeta actual.")

## 13.5 Abrir la app desde Colab (sin instalar nada en tu equipo)
Ejecuta esta celda y abre el enlace `https://….trycloudflare.com` que aparece, desde el celular o el computador. La app funciona mientras la celda esté en ejecución. Para detenerla, pulsa ⏹ en la celda.

In [ ]:
# ============================================================
# 13.5 LANZAR STREAMLIT + TÚNEL PÚBLICO
# ============================================================
!pip install -q streamlit==1.40.0
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

import subprocess, re, time
subprocess.run(['pkill', '-f', 'streamlit run'], check=False)
app_proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
                            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
tunel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for linea in tunel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', linea)
    if m:
        print(f"🌐 Abre la app aquí: {m.group(0)}")
        break
try:
    tunel.wait()
except KeyboardInterrupt:
    tunel.terminate(); app_proc.terminate(); print("App detenida.")